# 📝 Notes

## Before Running This Notebook:

1. **Install Dependencies:**
   ```bash
   pip install -r requirements.txt
   ```

2. **Setup Environment Variables:**
   - Copy `.env.example` to `.env`
   - Add your `DEEPSEEK_API_KEY`
   - Configure `WEAVIATE_URL` if needed

3. **Initialize Database:**
   ```bash
   python scripts/setup_database.py
   python scripts/ingest_documents.py
   ```

4. **Verify Weaviate is Running:**
   - Default: http://localhost:8080
   - Check schema is created

## Troubleshooting:

- **ModuleNotFoundError:** Make sure you're running from the notebooks directory
- **Weaviate Connection Error:** Check if Weaviate is running
- **Missing API Key:** Set `DEEPSEEK_API_KEY` in `.env`
- **No Results:** Ensure documents are indexed in Weaviate

In [4]:
# Cell 11: Quick Query Interface (No Verbose Output)
def quick_query(query: str):
    """
    Quick query with minimal output - just the answer
    
    Args:
        query: User query
    
    Returns:
        Answer string
    """
    print(f"❓ {query}\n")
    response = run_rag_pipeline(query, verbose=False)
    print(f"💡 {response.answer}\n")
    
    if response.citations:
        print(f"📚 Sources: {', '.join(response.citations[:3])}")
    
    return response.answer

# Example quick queries
print("🚀 Quick Query Mode:\n")
print("=" * 80)

quick_query("ควรแปรงฟันอย่างไรหลังผ่าตัด")
print("\n" + "-" * 80 + "\n")

quick_query("ยาแก้ปวดมีผลข้างเคียงอะไรบ้าง")

🚀 Quick Query Mode:

❓ ควรแปรงฟันอย่างไรหลังผ่าตัด



NameError: name 'run_rag_pipeline' is not defined

In [ ]:
# Cell 10: Inspect Document Details
def inspect_document(doc_result):
    """
    Display detailed information about a retrieved document
    
    Args:
        doc_result: SearchResult object
    """
    doc = doc_result.document
    
    print("=" * 80)
    print("📄 DOCUMENT DETAILS")
    print("=" * 80)
    print(f"ID: {doc.id}")
    print(f"Category: {doc.category}")
    print(f"Source File: {doc.source_file}")
    print(f"Page Number: {doc.page_number}")
    print(f"Relevance Score: {doc_result.score:.4f}")
    print(f"Rank: {doc_result.rank}")
    
    print("\n" + "-" * 80)
    print("CONTENT:")
    print("-" * 80)
    print(doc.content)
    
    if doc.metadata:
        print("\n" + "-" * 80)
        print("METADATA:")
        print("-" * 80)
        for key, value in doc.metadata.items():
            print(f"  {key}: {value}")
    
    print("=" * 80)

# Example: Inspect the top document from previous retrieval
if 'retrieval_result' in locals() and retrieval_result.reranked_results:
    print("Inspecting top document from last retrieval:\n")
    inspect_document(retrieval_result.reranked_results[0])
else:
    print("⚠️  No retrieval results available. Run a query first!")

# 🛠️ Utility Functions

## Inspect Retrieved Documents

In [ ]:
# Cell 9: Test Category-Specific Retrieval
def test_category_retrieval(query: str, category: str):
    """
    Test retrieval with specific category filter
    
    Args:
        query: User query
        category: Category to filter by
    """
    print(f"🏷️  Testing Category-Filtered Retrieval")
    print("=" * 80)
    print(f"Query: {query}")
    print(f"Category Filter: {category}")
    print("-" * 80)
    
    result = retrieval_service.retrieve_by_category(
        query_text=query,
        category=category,
        top_n=5
    )
    
    print(f"\n✅ Retrieved {len(result.reranked_results)} results from category '{category}'")
    print("\n📄 Results:")
    for i, res in enumerate(result.reranked_results, 1):
        doc = res.document
        preview = doc.content[:100].replace('\n', ' ')
        print(f"\n{i}. [Score: {res.score:.3f}]")
        print(f"   Content: {preview}...")
        print(f"   Category: {doc.category}")
        print(f"   Source: {doc.source_file}")
    
    return result

# Example: Test medication category
category_result = test_category_retrieval(
    query="ควรกินยาแก้ปวดอย่างไร",
    category="Medication"
)

In [ ]:
# Cell 8: Batch Testing Multiple Queries
def batch_test_queries(queries: list):
    """
    Test multiple queries and compare results
    
    Args:
        queries: List of query strings
    """
    print("🔄 Running Batch Test...")
    print("=" * 80)
    
    results = []
    for i, query in enumerate(queries, 1):
        print(f"\n📍 Query {i}/{len(queries)}: {query}")
        print("-" * 80)
        
        response = run_rag_pipeline(query, verbose=False)
        results.append({
            'query': query,
            'answer': response.answer,
            'citations': len(response.citations),
            'time_ms': response.generation_time_ms
        })
        
        # Show brief result
        answer_preview = response.answer[:150].replace('\n', ' ')
        print(f"✅ Answer: {answer_preview}...")
        print(f"   📝 Citations: {len(response.citations)}")
        print(f"   ⏱️  Time: {response.generation_time_ms:.2f} ms")
    
    print("\n" + "=" * 80)
    print("📊 Batch Test Summary:")
    print("=" * 80)
    avg_time = sum(r['time_ms'] for r in results) / len(results)
    avg_citations = sum(r['citations'] for r in results) / len(results)
    print(f"   Total Queries: {len(results)}")
    print(f"   Avg Time: {avg_time:.2f} ms")
    print(f"   Avg Citations: {avg_citations:.1f}")
    
    return results

# Example batch queries
batch_queries = [
    "หลังผ่าตัดควรทำอะไรบ้าง",
    "อาการบวมหลังผ่าตัดปกติหรือไม่",
    "ต้องงดอาหารประเภทไหนบ้าง"
]

batch_results = batch_test_queries(batch_queries)

# 🔬 Advanced Features

## Batch Processing & Category Filtering

In [ ]:
# Cell 7: Test Case 3 - Medication Instructions
response3 = run_rag_pipeline("ควรกินยาแก้ปวดตอนไหน")

In [ ]:
# Cell 6: Test Case 2 - Nutrition After Surgery
response2 = run_rag_pipeline("หลังผ่าตัดควรกินอาหารอะไร")

In [ ]:
# Cell 5: Test Case 1 - Post-operative Pain
response1 = run_rag_pipeline("ปวดฟันมากหลังผ่าตัด ต้องทำยังไง")

# 🧪 Test Cases

Below are example queries to test the RAG system with different types of dental/medical questions.

In [ ]:
# Cell 1: Setup Path & Autoreload
import sys
import os
from pathlib import Path

# เลื่อน Path ขึ้นไป 1 ชั้นเพื่อให้เจอ folder root project
project_root = str(Path(os.getcwd()).parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# สั่งให้ Notebook โหลด Code ใหม่ทันทีถ้าเราไปแก้ไฟล์ .py (ไม่ต้อง Restart Kernel)
%load_ext autoreload
%autoreload 2

print(f"✅ Project Root: {project_root}")
print(f"✅ Python Path: {sys.path[0]}")

In [ ]:
# Cell 2: Import Required Modules
import warnings
warnings.filterwarnings('ignore')

from config.settings import get_settings
from src.retrieval.service import RetrievalService
from src.generation.engine import GenerationEngine
from src.generation.client import DeepSeekClient
from src.generation.prompt_builder import PromptBuilder
from src.generation.response_parser import ResponseParser

print("✅ All modules imported successfully!")

In [ ]:
# Cell 3: Initialize System
print("🔧 Initializing RAG System...")
print("-" * 60)

# Load settings
settings = get_settings()
print(f"✅ Settings loaded")
print(f"   - Weaviate URL: {settings.weaviate_url}")
print(f"   - DeepSeek API: {'configured' if settings.deepseek_api_key else 'missing'}")

# Initialize Retrieval Service
print("\n🔍 Initializing Retrieval Service...")
try:
    retrieval_service = RetrievalService(settings)
    print(f"✅ Retrieval Service ready")
    print(f"   - Alpha: {retrieval_service.alpha}")
    print(f"   - Top-K: {retrieval_service.top_k}")
    print(f"   - Top-N: {retrieval_service.top_n}")
except Exception as e:
    print(f"❌ Retrieval Service Error: {e}")
    raise

# Initialize Generation Engine
print("\n🤖 Initializing Generation Engine...")
try:
    llm_client = DeepSeekClient(
        api_key=settings.deepseek_api_key,
        model_name=settings.model_config['llm']['model_name'],
        temperature=settings.model_config['llm']['temperature'],
        max_tokens=settings.model_config['llm']['max_tokens']
    )
    
    prompt_builder = PromptBuilder(settings.prompts)
    response_parser = ResponseParser()
    
    generation_engine = GenerationEngine(
        llm_client=llm_client,
        prompt_builder=prompt_builder,
        response_parser=response_parser,
        min_context_score=0.5,
        min_context_count=1
    )
    
    print(f"✅ Generation Engine ready")
    print(f"   - Model: {llm_client.model_name}")
    print(f"   - Temperature: {llm_client.temperature}")
    print(f"   - Max Tokens: {llm_client.max_tokens}")
except Exception as e:
    print(f"❌ Generation Engine Error: {e}")
    raise

print("\n" + "=" * 60)
print("🎉 System Initialized Successfully!")
print("=" * 60)

In [ ]:
# Cell 4: Define RAG Pipeline Function
def run_rag_pipeline(query_text: str, verbose: bool = True):
    """
    Run complete RAG pipeline: Retrieval → Generation → Display Results
    
    Args:
        query_text: User query in Thai
        verbose: Whether to print detailed logs
    
    Returns:
        GeneratedResponse object
    """
    if verbose:
        print("=" * 80)
        print(f"❓ User Query: {query_text}")
        print("=" * 80)
    
    # Step 1: Retrieve relevant documents
    if verbose:
        print("\n🔍 STEP 1: Retrieving Context...")
        print("-" * 80)
    
    retrieval_result = retrieval_service.retrieve(query_text)
    
    if verbose:
        print(f"✅ Retrieved {len(retrieval_result.candidates)} candidates")
        print(f"✅ Reranked to top {len(retrieval_result.reranked_results)} results")
        print(f"⏱️  Retrieval Time: {retrieval_result.retrieval_time_ms:.2f} ms")
        
        # Show top results
        print("\n📄 Top Retrieved Documents:")
        for i, result in enumerate(retrieval_result.reranked_results[:3], 1):
            doc = result.document
            preview = doc.content[:100].replace('\n', ' ')
            print(f"   {i}. [Score: {result.score:.3f}] {preview}...")
            print(f"      Category: {doc.category} | Source: {doc.source_file}")
    
    # Step 2: Generate response
    if verbose:
        print("\n🤖 STEP 2: Generating Response...")
        print("-" * 80)
    
    response = generation_engine.generate(retrieval_result, query_text)
    
    if verbose:
        print(f"✅ Response generated")
        print(f"⏱️  Generation Time: {response.generation_time_ms:.2f} ms")
    
    # Step 3: Display results
    if verbose:
        print("\n" + "=" * 80)
        print("💡 ANSWER:")
        print("=" * 80)
        print(response.answer)
        
        print("\n" + "-" * 80)
        print("📚 CITATIONS:")
        print("-" * 80)
        if response.citations:
            for i, cite in enumerate(response.citations, 1):
                print(f"   [{i}] {cite}")
        else:
            print("   (No citations)")
        
        # Summary
        total_time = retrieval_result.retrieval_time_ms + response.generation_time_ms
        print("\n" + "=" * 80)
        print("📊 SUMMARY:")
        print("=" * 80)
        print(f"   ⏱️  Total Time: {total_time:.2f} ms")
        print(f"   📄 Documents Used: {len(response.context_used)}")
        print(f"   📝 Citations: {len(response.citations)}")
        print(f"   🤖 Model: {response.model_name}")
        print("=" * 80)
    
    return response

print("✅ RAG Pipeline function defined successfully!")